# 02 · TrOCR ile Türkçe El Yazısı İnce Ayarı
`microsoft/trocr-base-handwritten` (MIT) satır-düzeyi modeli, 01'de üretilen sentetik veriyle Türkçe'ye uyarlanır. **Ücretsiz T4'e sığar.** Çıktı Mac'te `--taniyici trocr --model <dizin>` ile kullanılır.

> Sentetik ön-eğitimden sonra AZ SAYIDA (16–256) gerçek el yazısı satırla ikinci bir ince ayar CER'i belirgin düşürür (arXiv 2503.19546).

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
!pip install -q transformers datasets jiwer evaluate accelerate
!git clone https://github.com/KULLANICI/sinav-okur.git  # <-- kendi repo URL'nizle değiştirin || true
%cd sinav-okur && pip install -q -e . && cd ..

In [ ]:
# Veriyi getir (01 çıktısı)
!unzip -qo /content/drive/MyDrive/sinav_okur_veri.zip -d /content
import json
kayitlar = [json.loads(l) for l in open('/content/veri/labels.jsonl')]
print(len(kayitlar), 'satır')

In [ ]:
from datasets import Dataset, Image as HfImage
import os
for k in kayitlar: k['image'] = os.path.join('/content/veri', k['image'])
ds = Dataset.from_list(kayitlar).cast_column('image', HfImage())
ds = ds.train_test_split(test_size=0.05, seed=1)
print(ds)

In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
import torch
AD = 'microsoft/trocr-base-handwritten'
islemci = TrOCRProcessor.from_pretrained(AD)
model = VisionEncoderDecoderModel.from_pretrained(AD)
# decoder başlangıç/padding token ayarları
model.config.decoder_start_token_id = islemci.tokenizer.cls_token_id
model.config.pad_token_id = islemci.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size
model.config.max_length = 96

> **Not (Türkçe token'ları):** TrOCR'ın byte-level BPE tokenizer'ı Türkçe karakterleri byte dizileriyle temsil eder; karakter kaybı olmaz ama nadir harfler daha çok token'a bölünür. Sentetik veri hacmi bunu telafi eder. CER hedefe ulaşmazsa decoder'ı `dbmdz/bert-base-turkish-cased` ile değiştirmeyi deneyin.

In [ ]:
def haz(ornek):
    px = islemci(ornek['image'].convert('RGB'), return_tensors='pt').pixel_values[0]
    etiket = islemci.tokenizer(ornek['text'], padding='max_length',
                               max_length=96, truncation=True).input_ids
    etiket = [t if t != islemci.tokenizer.pad_token_id else -100 for t in etiket]
    return {'pixel_values': px, 'labels': etiket}
hazir = ds.map(haz, remove_columns=ds['train'].column_names)

In [ ]:
import evaluate
cer = evaluate.load('cer')
def metrik(p):
    tah = islemci.batch_decode(p.predictions, skip_special_tokens=True)
    import numpy as np
    etik = np.where(p.label_ids != -100, p.label_ids, islemci.tokenizer.pad_token_id)
    ger = islemci.batch_decode(etik, skip_special_tokens=True)
    return {'cer': cer.compute(predictions=tah, references=ger)}

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, default_data_collator
args = Seq2SeqTrainingArguments(
    output_dir='trocr-tr', per_device_train_batch_size=8,
    per_device_eval_batch_size=8, predict_with_generate=True, fp16=True,
    num_train_epochs=3, learning_rate=5e-5, eval_strategy='steps',
    save_steps=1000, eval_steps=1000, logging_steps=100, save_total_limit=2,
    report_to='none')
trainer = Seq2SeqTrainer(model=model, args=args, compute_metrics=metrik,
    train_dataset=hazir['train'], eval_dataset=hazir['test'],
    data_collator=default_data_collator, tokenizer=islemci.feature_extractor)
trainer.train()

In [ ]:
# Kaydet + Drive + Mac'e indir
model.save_pretrained('trocr-tr'); islemci.save_pretrained('trocr-tr')
!zip -qr /content/drive/MyDrive/trocr-tr.zip trocr-tr
print('Mac: sinav-okur isle ... --taniyici trocr --model ./trocr-tr')